# 01 · Data audit

**This is where the project most likely goes quietly wrong.** The benign-set
provenance decision made here determines the result, and it is the first thing
a hostile reviewer will attack.

Before implementing any loader, the corresponding decision record must be
filled in:

- `docs/decisions/0002-drebin-benign-corpus.md`
- `docs/decisions/0003-cicmaldroid-riskware.md`
- `docs/decisions/0004-androzoo-temporal.md`

**Blocking task:** request the AndroZoo API key now. It takes days to weeks and
gates both the temporal axis and the Drebin benign corpus.


In [ ]:
# Bootstrap -- see environment/colab_bootstrap.md for the full version
import os
os.environ.setdefault('AFS_DATA_ROOT', '/content/drive/MyDrive/afs-data')
os.environ.setdefault('AFS_SCRATCH', '/content/afs-scratch')
from afs.paths import Paths
from afs.config import resolve_experiment
from afs.pipeline import run_experiment
paths = Paths.create()
print('data root:', paths.data_root)


## Raw data integrity

Hash every file at download; verify before each stage. A partially-downloaded dataset you then train on for three weeks is the failure this prevents.


In [ ]:
from afs.utils.hashing import hash_file
import json, pathlib
manifest = {str(p.relative_to(paths.raw)): hash_file(p)
            for p in sorted(paths.raw.rglob('*')) if p.is_file()}
print(json.dumps(manifest, indent=2)[:800])
# Commit this to git -- it is a record, not an artifact.
pathlib.Path('docs/raw_checksums.json').write_text(json.dumps(manifest, indent=2))

## Loader implementation

Implement in `src/afs/data/registry.py`, not here. This cell only exercises it.


In [ ]:
from afs.data import load_dataset
ds = load_dataset('synthetic', paths.raw)   # swap once loaders land
import json; print(json.dumps(ds.summary(), indent=2, default=str))

## Provenance table

This table goes in the paper. If you cannot fill it in, the loader is not done.


In [ ]:
import pandas as pd
rows = [{'sample': k, **vars(v)} for k, v in list(ds.provenance.items())[:5]]
display(pd.DataFrame(rows))
print(pd.Series([p.source for p in ds.provenance.values()]).value_counts())